In [17]:
# train_code_chunker.py

import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertForTokenClassification, get_scheduler
from torch.optim import AdamW

# 1) CONFIG
MODEL_NAME = "distilbert-base-uncased"
LABEL_LIST = ["O", "B-CODE", "I-CODE"]
LABEL2ID = {l:i for i,l in enumerate(LABEL_LIST)}
ID2LABEL = {i:l for l,i in LABEL2ID.items()}

TRAIN_FILE = "train.jsonl"
DEV_FILE   = "dev.jsonl"
BATCH_SIZE = 8
EPOCHS     = 30
LR         = 5e-5
MAX_LEN    = 128
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2) DATASET
class CodeChunkDataset(Dataset):
    def __init__(self, path, tokenizer):
        self.examples = []
        with open(path) as f:
            for line in f:
                obj = json.loads(line)
                tokens, labels = obj["tokens"], obj["labels"]
                enc = tokenizer(tokens,
                                is_split_into_words=True,
                                truncation=True,
                                padding="max_length",
                                max_length=MAX_LEN,
                                return_offsets_mapping=True)
                word_ids = enc.word_ids()
                label_ids = []
                previous_word_idx = None
                for i, word_idx in enumerate(word_ids):
                  if word_idx is None:
                      label_ids.append(-100)  # special tokens or padding
                  elif word_idx >= len(labels):
                      label_ids.append(-100)  # invalid index, skip
                  elif word_idx != word_ids[i - 1]:
                      label_ids.append(LABEL2ID[labels[word_idx]])
                  else:
                      lab = labels[word_idx]
                      label_ids.append(LABEL2ID["I-" + lab[2:]] if lab.startswith("B-") else LABEL2ID[lab])


                self.examples.append({
                    "input_ids": torch.tensor(enc["input_ids"]),
                    "attention_mask": torch.tensor(enc["attention_mask"]),
                    "labels": torch.tensor(label_ids)
                })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

# 3) PREPARE
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
train_ds = CodeChunkDataset(TRAIN_FILE, tokenizer)
dev_ds   = CodeChunkDataset(DEV_FILE, tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
dev_loader   = DataLoader(dev_ds, batch_size=BATCH_SIZE)

# 4) MODEL
model = DistilBertForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS

scheduler = get_scheduler("linear",
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# 5) TRAINING LOOP
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Train loss: {avg_train_loss:.4f}")

    model.eval()
    eval_loss = 0
    with torch.no_grad():
        for batch in dev_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)
            eval_loss += outputs.loss.item()
            # print(outputs)
    print(f"Epoch {epoch+1} Dev loss: {eval_loss/len(dev_loader):.4f}")

# 6) SAVE
model.save_pretrained("code-chunker-distilbert")
tokenizer.save_pretrained("code-chunker-distilbert")
print("Model saved to code-chunker-distilbert/")


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 Train loss: 0.6763
Epoch 1 Dev loss: 0.3490
Epoch 2 Train loss: 0.3232
Epoch 2 Dev loss: 0.1297
Epoch 3 Train loss: 0.1875
Epoch 3 Dev loss: 0.0642
Epoch 4 Train loss: 0.1399
Epoch 4 Dev loss: 0.0367
Epoch 5 Train loss: 0.0915
Epoch 5 Dev loss: 0.0110
Epoch 6 Train loss: 0.0675
Epoch 6 Dev loss: 0.0056
Epoch 7 Train loss: 0.0475
Epoch 7 Dev loss: 0.0077
Epoch 8 Train loss: 0.0565
Epoch 8 Dev loss: 0.0025
Epoch 9 Train loss: 0.0416
Epoch 9 Dev loss: 0.0022
Epoch 10 Train loss: 0.0244
Epoch 10 Dev loss: 0.0017
Epoch 11 Train loss: 0.0268
Epoch 11 Dev loss: 0.0015
Epoch 12 Train loss: 0.0227
Epoch 12 Dev loss: 0.0014
Epoch 13 Train loss: 0.0182
Epoch 13 Dev loss: 0.0011
Epoch 14 Train loss: 0.0135
Epoch 14 Dev loss: 0.0010
Epoch 15 Train loss: 0.0121
Epoch 15 Dev loss: 0.0009
Epoch 16 Train loss: 0.0093
Epoch 16 Dev loss: 0.0008
Epoch 17 Train loss: 0.0055
Epoch 17 Dev loss: 0.0007
Epoch 18 Train loss: 0.0051
Epoch 18 Dev loss: 0.0007
Epoch 19 Train loss: 0.0046
Epoch 19 Dev loss:

In [18]:
import torch
from transformers import DistilBertTokenizerFast, DistilBertForTokenClassification

LABEL_LIST = ["O", "B-CODE", "I-CODE"]
ID2LABEL = {i: l for i, l in enumerate(LABEL_LIST)}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = DistilBertTokenizerFast.from_pretrained("/content/code-chunker-distilbert")
model = DistilBertForTokenClassification.from_pretrained("/content/code-chunker-distilbert").to(DEVICE)

def predict_spans(text):
    enc = tokenizer(text.split(), is_split_into_words=True, return_tensors="pt",
                    truncation=True, padding="max_length", max_length=128)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    logits = model(**enc).logits
    preds = logits.argmax(-1)[0].cpu().tolist()
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

    spans = []
    current = None
    for tok, lab_id in zip(tokens, preds):
        if tok in ["[PAD]", "[CLS]", "[SEP]"]:
            continue  # skip irrelevant tokens
        lab = ID2LABEL[lab_id]
        if lab.startswith("B-"):
            if current: spans.append(current)
            current = {"type": lab[2:], "tokens": [tok]}
        elif lab.startswith("I-") and current:
            current["tokens"].append(tok)
        else:
            if current:
                spans.append(current)
                current = None
    if current:
        spans.append(current)
    return spans


print(predict_spans("for i in range(n): print(i)"))


[{'type': 'CODE', 'tokens': ['for', 'i', 'in', 'range', '(', 'n', ')', ':', 'print', '(', 'i', ')']}]


In [34]:
from transformers import DistilBertTokenizerFast, DistilBertForTokenClassification
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_LIST = ["O", "B-CODE", "I-CODE"]
ID2LABEL = {i: l for i, l in enumerate(LABEL_LIST)}

# Load your fine-tuned model
tokenizer = DistilBertTokenizerFast.from_pretrained("/content/code-chunker-distilbert")
model = DistilBertForTokenClassification.from_pretrained("/content/code-chunker-distilbert").to(DEVICE)

def predict_spans(text):
    # Tokenize
    enc = tokenizer(text.split(), is_split_into_words=True, return_tensors="pt",
                    truncation=True, padding="max_length", max_length=128)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    # Predict
    with torch.no_grad():
        logits = model(**enc).logits
    predictions = torch.argmax(logits, dim=-1)[0].cpu().tolist()

    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    word_ids = enc['input_ids'][0].cpu().tolist()

    spans = []
    current = None
    for tok, lab_id in zip(tokens, predictions):
        lab = ID2LABEL[lab_id]
        if tok in ["[PAD]", "[CLS]", "[SEP]"]:
            continue  # skip irrelevant tokens
        if lab.startswith("B-"):
            if current:
                spans.append(current)
            current = {"type": lab[2:], "tokens": [tok]}
        elif lab.startswith("I-") and current:
            current["tokens"].append(tok)
        else:
            if current:
                spans.append(current)
                current = None
    if current:
        spans.append(current)

    # Pretty Print
    print("\n--- CODE SPANS FOUND ---")
    for i, span in enumerate(spans):
        joined = tokenizer.convert_tokens_to_string(span['tokens']).replace(' ##', '')
        print(f"[{i+1}] Type: {span['type']} | Code: {joined}")
    print("--- END ---\n")

    return spans


# Example usage
text = '''
Packages algorithm, algorithmicx and algpseudocode are used for setting algorithms in LATEX using the format:

\\begin{algorithm}
\\caption{<alg-caption>}\label{<alg-label>}
\\begin{algorithmic}[1]
\\end{algorithmic}
\\end{algorithm}

You may refer above listed package documentations for more details before setting algorithm environment. For program codes, the “program” package is required and the command to be used
'''
spans = predict_spans(text)


--- CODE SPANS FOUND ---
[1] Type: CODE | Code: algorithmicx and algpseudocode
[2] Type: CODE | Code: \ begin { algorithm }
[3] Type: CODE | Code: \ caption { < alg - caption > }
[4] Type: CODE | Code: \ label { < alg - label > }
[5] Type: CODE | Code: \ begin { algorithmic } [ 1 ]
[6] Type: CODE | Code: \ end
[7] Type: CODE | Code: { algorithmic } \ end { algorithm }
[8] Type: CODE | Code: “ program ” package
--- END ---

